# Project 02: VQE for Molecular Ground State (H2)

This notebook builds an end-to-end VQE pipeline for the H2 molecule.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.circuit.library import TwoLocal

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit_nature.second_q.algorithms import NumPyMinimumEigensolverFactory

## 1) Define Molecule

In [ ]:
bond_length = 0.735  # Angstrom
driver = PySCFDriver(
    atom=f"H 0 0 0; H 0 0 {bond_length}",
    basis="sto3g",
)
problem = driver.run()

# Optional: reduce active space for small systems
transformer = ActiveSpaceTransformer(num_electrons=2, num_spatial_orbitals=2)
problem = transformer.transform(problem)

## 2) Build Qubit Hamiltonian

In [ ]:
mapper = JordanWignerMapper()
hamiltonian = problem.hamiltonian.second_q_op()
qubit_hamiltonian = mapper.map(hamiltonian)

print(qubit_hamiltonian)

## 3) Exact Reference Energy

In [ ]:
exact_solver = GroundStateEigensolver(mapper, NumPyMinimumEigensolverFactory())
exact_result = exact_solver.solve(problem)
exact_energy = exact_result.total_energies[0].real
print(f"Exact ground energy: {exact_energy:.8f}")

## 4) VQE Setup

In [ ]:
num_qubits = qubit_hamiltonian.num_qubits
ansatz = TwoLocal(num_qubits, ["ry", "rz"], "cz", reps=2)
optimizer = SLSQP(maxiter=200)
estimator = StatevectorEstimator()

energies = []

def callback(eval_count, params, mean, std):
    energies.append(mean)

vqe = VQE(
    estimator=estimator,
    ansatz=ansatz,
    optimizer=optimizer,
    callback=callback,
)

## 5) Run VQE + Compare

In [ ]:
vqe_result = vqe.compute_minimum_eigenvalue(qubit_hamiltonian)
vqe_energy = vqe_result.eigenvalue.real

print(f"VQE energy:   {vqe_energy:.8f}")
print(f"Exact energy: {exact_energy:.8f}")
print(f"Absolute error: {abs(vqe_energy - exact_energy):.8e}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(energies, marker="o", linewidth=1)
plt.axhline(exact_energy, color="red", linestyle="--", label="Exact")
plt.title("VQE Convergence for H2")
plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()